# Adding retrieval to enriched documents

*Chunks are derived documents too — and a search index cannot join, which is a modelling decision rather than a limitation to route around.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omnifroodle/couchbase_notebooks/blob/main/notebooks/data-model/02_adding_retrieval.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/omnifroodle/couchbase_notebooks?quickstart=1)

**Claim.** Filtering a vector search by a field that lives on the parent document forces a choice: copy the field down, or filter afterwards and lose results.
**Result.** A filter on a field the index does not carry returns **zero hits and no error**. Copy three fields onto the chunks and the same query answers *"confidentiality terms, California law, signed since 2015"* in one request.
**Requires.** couchbase · local-embeddings
**Read** ~10 min · **Run** ~5 min · **Cost** $0.00 — no LLM calls

[`data-model/01`](01_documents_that_learn.ipynb) gave twenty contracts a set of extracted terms
and queried them with SQL++. This notebook makes the same documents searchable by *meaning*,
which sounds like more of the same and is not.

A contract is thirty thousand characters. Too long to embed as one vector — you would be
averaging a termination clause with a payment schedule and getting something that means
neither — so the searchable unit has to be a **chunk**. And chunks are exactly the shape `01`
called a *derived document*: records that exist because of a source, keyed from it, and that
nobody would suggest storing inside it.

That split creates the problem this notebook is about. The terms are on the contract. The
vectors are on the chunks. And a search index **cannot join them**.

**This notebook starts by getting it wrong**, because the failure is silent and worth seeing
once.

In [ ]:
# --- Setup. Works in a local checkout and on Colab. -------------------------
import os
import pathlib
import subprocess
import sys

# Cloned on Colab, where there is no local checkout. Override to test a fork.
REPO_URL = os.environ.get("CBNB_REPO_URL", "https://github.com/omnifroodle/couchbase_notebooks")

try:
    import cbnb
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((p for p in [here, *here.parents] if (p / "cbnb" / "__init__.py").exists()), None)
    if root is None:
        # Colab: clone the repo so the committed datasets come with it.
        subprocess.check_call(["git", "clone", "--depth", "1", "--quiet", REPO_URL, "cbnb-repo"])
        root = pathlib.Path("cbnb-repo").resolve()
    sys.path.insert(0, str(root))
    import cbnb

settings = cbnb.bootstrap(requires=["couchbase", "local-embeddings"])

## 1. What notebook 01 left behind

The extracted terms are already in the database, as their own documents. Nothing here re-runs
a model — this notebook costs nothing to run, because the expensive part already happened and
was **stored**.

That is the point of the series: enrichment is not a step in a pipeline you re-run, it is a
property the documents now have.

In [2]:
import pandas as pd

from cbnb.couchbase_io import connect

BUCKET, SCOPE = settings.cb_bucket, "contract_intelligence"
CONTRACTS, TERMS, CHUNKS = "contracts", "contract_terms", "chunks"
INDEX = "contract_chunk_search"

cluster = connect(settings)

terms = pd.DataFrame(list(cluster.query(f"""
    SELECT contract_id, title, governing_law, agreement_date, term_years, auto_renews
    FROM `{BUCKET}`.`{SCOPE}`.`{TERMS}`
""")))

if terms.empty:
    raise RuntimeError(
        "No extracted terms found. Run data-model/01 first -- this notebook reads what it wrote."
    )

print(f"{len(terms)} contracts with extracted terms")
terms.head(4)

20 contracts with extracted terms


,agreement_date,auto_renews,contract_id,governing_law,title,term_years
0,2014-01-24T00:00:00Z,False,1,People's Republic of China,COOPERATION AGREEMENT (2014 Amendment),NaN
1,1996-11-05T00:00:00Z,True,11,California,DISTRIBUTOR AGREEMENT,1.0
2,2010-07-15T00:00:00Z,True,18,New York,Strategic Alliance Agreement,5.0
3,1999-06-30T00:00:00Z,False,17,New York,ONLINE HOSTING AGREEMENT,1.0


## 2. Chunks, as derived documents

Same key convention as the terms records: `contract::0::chunk::7` belongs to `contract::0`
because of how it is named.

Deliberately naive to begin with. Each chunk carries its text, its position, its embedding, and
the id of the contract it came from — and nothing else.

In [3]:
from cbnb.couchbase_io import ensure_collection, upsert_docs
from cbnb.embeddings import Embedder

CHUNK_CHARS, OVERLAP = 1200, 200
embedder = Embedder()

sources = {
    row["contract_id"]: row["text"]
    for row in cluster.query(
        f"SELECT contract_id, text FROM `{BUCKET}`.`{SCOPE}`.`{CONTRACTS}`")
}

chunks = []
for contract_id, text in sorted(sources.items()):
    for n, start in enumerate(range(0, max(len(text) - OVERLAP, 1), CHUNK_CHARS - OVERLAP)):
        chunks.append({
            "key": f"contract::{contract_id}::chunk::{n}",
            "type": CHUNKS,
            "of": f"contract::{contract_id}",
            "contract_id": contract_id,
            "position": n,
            "text": text[start:start + CHUNK_CHARS],
        })

vectors = embedder.encode([c["text"] for c in chunks])
for chunk, vector in zip(chunks, vectors):
    chunk["embedding"] = vector.tolist()

chunk_docs = ensure_collection(cluster, BUCKET, SCOPE, CHUNKS)
upsert_docs(chunk_docs, {c["key"]: c for c in chunks}, progress=False)
print(f"{len(chunks):,} chunks from {len(sources)} contracts")

609 chunks from 20 contracts


In [4]:
from cbnb.couchbase_io import ensure_vector_index, wait_for_index

ensure_vector_index(
    cluster,
    bucket_name=BUCKET, scope_name=SCOPE, collection_name=CHUNKS,
    index_name=INDEX,
    vector_field="embedding", dims=embedder.dims,
    text_fields=["text"],
)
wait_for_index(cluster, bucket_name=BUCKET, scope_name=SCOPE, index_name=INDEX,
               expected=len(chunks))

  indexed 0/609 (not serving yet)   

  indexed 604/609 (indexing)   

  indexed 609/609 (definition just changed)   

  indexed 609/609 (definition just changed)   

  indexed 609/609 (ready)   

  indexed 609/609 (ready)   

609

## 3. The filter that matches nothing

Now ask the question this series has been building towards: *confidentiality obligations, in
contracts governed by California law.*

`governing_law` is on the contract. The search is over chunks.

In [5]:
import couchbase.search as search

from cbnb.couchbase_io import vector_search

QUESTION = "confidentiality and non-disclosure obligations"
where = dict(bucket_name=BUCKET, scope_name=SCOPE, index_name=INDEX)

hits = vector_search(
    cluster, **where, vector_field="embedding",
    query_vector=embedder.encode_one(QUESTION),
    k=5, num_candidates=400, fields=["text"],
    prefilter=search.TermQuery("California", field="governing_law"),
)
print(f"hits: {len(hits)}")

hits: 0


Zero hits. No exception, no warning, no "unknown field" — an empty result set, which reads
exactly like *"no contracts match"*.

It is not that no contracts match. Three of them are governed by California law, and `01`
printed that table. The index simply has no `governing_law` field to filter on, so the filter
excluded everything.

This is the most expensive failure mode in the whole repo, because nothing tells you about it.
The diagnosis is one call.

In [6]:
from cbnb.couchbase_io import indexed_fields

indexed_fields(cluster, **where)

{'embedding': 'vector', 'text': 'text (en)'}

`text` and `embedding`. That is all the index covers. A `TermQuery` on `governing_law` was
always going to match nothing.

## 4. A search index cannot join

The terms are on `contract::0`. The vectors are on `contract::0::chunk::7`. SQL++ would join
those in a line — `01` did exactly that with `ON KEYS`. A search index will not, because it
scores one document at a time against one query, and there is no step at which it could go and
fetch a parent.

So there are two honest options, and they are a trade rather than a right answer.

**Copy the parent's fields onto the children.** Denormalisation, on purpose. The index can then
filter and search in one pass, and you get exactly the `k` results you asked for. The cost is
that the terms now live in two places, so re-extracting a contract means rewriting its chunks.

**Filter afterwards.** Retrieve widely, look up each hit's parent, discard the misses. Nothing
is duplicated and the terms stay in one place — but you no longer control how many results you
end up with, and for a selective filter the answer is frequently "almost none".

Try the second one first, since it needs nothing built.

In [7]:
law_by_contract = terms.set_index("contract_id").governing_law.to_dict()

wide = vector_search(
    cluster, **where, vector_field="embedding",
    query_vector=embedder.encode_one(QUESTION),
    k=50, num_candidates=400, fields=["text"],
)
signed = pd.to_datetime(terms.set_index("contract_id").agreement_date, errors="coerce")


def parent(hit):
    return int(hit.id.split("::")[1])


for label, keep in [
    ("California", lambda h: law_by_contract.get(parent(h)) == "California"),
    ("California AND signed since 2015",
     lambda h: law_by_contract.get(parent(h)) == "California"
     and pd.notna(signed.get(parent(h))) and signed[parent(h)].year >= 2015),
]:
    kept = [h for h in wide if keep(h)]
    print(f"{label:34} retrieved {len(wide)}, kept {len(kept):2d}, "
          f"asked for 5 -> got {min(len(kept), 5)}")

California                         retrieved 50, kept  7, asked for 5 -> got 5
California AND signed since 2015   retrieved 50, kept  5, asked for 5 -> got 5


Watch the slack disappear. The loose filter keeps seven of fifty and returns the five you
asked for. Add one condition and exactly five survive — still five returned, with nothing to
spare. A third condition, or a less common jurisdiction, and this quietly starts handing back
three.

That is the shape of the problem rather than a catastrophe — and it is why the trade is a trade.
Post-filtering works until the filter is selective, and then it degrades in a way you only
notice by counting: the query still returns *something*, just not as much as you asked for.
Widen `k` until it is reliably enough and you are reading a growing share of the collection to
answer one question.
It is the same prefilter-versus-post-filter trade
[`retrieval/01`](../retrieval/01_building_hybrid_search.ipynb) introduces, with a real cost
attached.

## 5. Copy down only what you filter on

Three fields, chosen because queries filter on them. Not the parties, not the title, not the
full extraction — those stay on the terms document, where there is one copy to keep right.

**Copy what you filter by; reference what you display.** A hit gives you `contract_id`, and one
key lookup gets everything else at read time.

In [8]:
def rfc3339(value):
    parsed = pd.to_datetime(value, errors="coerce")
    return None if pd.isna(parsed) else parsed.strftime("%Y-%m-%dT%H:%M:%SZ")


def text_or(value, fallback):
    """`value or fallback` is wrong here: bool(float("nan")) is True, so a missing
    field would sail through as NaN and fail to encode as JSON."""
    return value if pd.notna(value) and value else fallback


filterable = {
    int(row.contract_id): {
        "governing_law": text_or(row.governing_law, "unstated"),
        "agreement_date": rfc3339(row.agreement_date),
        "term_years": float(row.term_years) if pd.notna(row.term_years) else 0.0,
    }
    for row in terms.itertuples(index=False)
}

import couchbase.subdocument as SD

for chunk in chunks:
    parent = filterable.get(chunk["contract_id"], {})
    chunk_docs.mutate_in(chunk["key"], [
        SD.upsert(name, value) for name, value in parent.items() if value is not None
    ])

print(f"copied {len(filterable[0])} filterable fields onto {len(chunks):,} chunks")
print("chunk documents are otherwise unchanged -- the text and the vector were not rewritten")

copied 3 filterable fields onto 609 chunks
chunk documents are otherwise unchanged -- the text and the vector were not rewritten


In [9]:
ensure_vector_index(
    cluster,
    bucket_name=BUCKET, scope_name=SCOPE, collection_name=CHUNKS,
    index_name=INDEX,
    vector_field="embedding", dims=embedder.dims,
    text_fields=["text"],
    keyword_fields=["governing_law"],
    numeric_fields=["term_years"],
    datetime_fields=["agreement_date"],
)
wait_for_index(cluster, bucket_name=BUCKET, scope_name=SCOPE, index_name=INDEX,
               expected=len(chunks))

indexed_fields(cluster, **where)

  indexed 0/609 (not serving yet)   

  indexed 609/609 (definition just changed)   

  indexed 609/609 (definition just changed)   

  indexed 609/609 (definition just changed)   

  indexed 609/609 (ready)   

  indexed 609/609 (ready)   

{'agreement_date': 'datetime',
 'embedding': 'vector',
 'governing_law': 'text (keyword)',
 'term_years': 'number',
 'text': 'text (en)'}

## 6. One request, three kinds of matching

*"What do agreements under California law, signed since 2015, say about confidentiality?"*

Three different questions in one sentence:

| part | matched by | field type |
| --- | --- | --- |
| confidentiality | meaning | vector |
| California | an exact value | keyword |
| since 2015 | a range | datetime |

None of those fields existed before `01` ran. They were prose.

In [10]:
selective = search.ConjunctionQuery(
    search.TermQuery("California", field="governing_law"),
    search.DateRangeQuery(start="2015-01-01T00:00:00Z", field="agreement_date"),
)

hits = vector_search(
    cluster, **where, vector_field="embedding",
    query_vector=embedder.encode_one(QUESTION),
    k=5, num_candidates=400,
    fields=["text", "governing_law", "agreement_date"],
    prefilter=selective,
)

print(f"{len(hits)} chunks matched all three conditions\n")
for hit in hits:
    contract_id = hit.id.split("::")[1]
    print(f"  contract {contract_id} · {hit['governing_law']} · {hit['agreement_date'][:10]}")
    print(f"    {' '.join(hit['text'].split())[:140]}\n")

5 chunks matched all three conditions

  contract 2 · California · 2018-02-01
    d as "Information" in the Non- Disclosure Agreement between the parties dated August 24, 2017, and accordingly the restrictions relating to 

  contract 2 · California · 2018-02-01
    ublicity, personal or proprietary right, or other common law or statutory right, nor defame any person or entity in the United States and Eu

  contract 8 · California · 2017-04-28
    country and (c) at Imprimis' expense, to assist Imprimis in prosecuting any such rights. Page 2 of 11 5.4. Surgical agrees that promptly upo

  contract 2 · California · 2018-02-01
    Other Charges or Expenses. Neither party will be liable to pay the other party any other types of charges or expenses not agreed to in this 

  contract 8 · California · 2017-04-28
    ability and Accountability Act of 1996 ("HIPAA"). 7. Conflicts of Interest. 7.1. Surgical represents and warrants that Surgical is not under



In [11]:
# What each condition is doing, one at a time.
for label, prefilter in [
    ("no filter", None),
    ("California", search.TermQuery("California", field="governing_law")),
    ("signed since 2015", search.DateRangeQuery(start="2015-01-01T00:00:00Z",
                                                field="agreement_date")),
    ("a fixed term of 3+ years", search.NumericRangeQuery(min=3, field="term_years")),
    ("California AND since 2015", selective),
]:
    found = vector_search(
        cluster, **where, vector_field="embedding",
        query_vector=embedder.encode_one(QUESTION),
        k=20, num_candidates=400, fields=["governing_law"], prefilter=prefilter,
    )
    contracts_hit = sorted({h.id.split("::")[1] for h in found})
    print(f"{label:26} {len(found):2d} chunks from contracts {contracts_hit}")

no filter                  20 chunks from contracts ['0', '13', '14', '17', '18', '2', '3', '4', '5', '7']


California                 20 chunks from contracts ['11', '2', '8']
signed since 2015          20 chunks from contracts ['0', '13', '2', '3', '4', '7', '8']


a fixed term of 3+ years   20 chunks from contracts ['18', '2', '8']


California AND since 2015  20 chunks from contracts ['2', '8']


## What it cost

The trade was made deliberately, so it is worth naming the bill.

**The terms are now in two places.** `contract::0::terms` holds the extraction;
`contract::0::chunk::*` each hold three of its fields. Re-extracting one contract means
rewriting its chunks as well — the marker field from `01` tells you which, so it is a targeted
update rather than a rebuild.

**Only three fields, though.** Everything else stays single-copy on the terms document. The
smaller the copied-down set, the cheaper the staleness problem, which is why the rule is *copy
what you filter by, reference what you display*.

**And the chunks did not need rewriting to gain them.** Subdocument mutation added three paths
to each chunk without touching the text or the 384-dimension vector sitting beside them.

For contract terms — extracted once, changing almost never — this is an easy trade. For a field
that changes every minute it would not be, and post-filtering would earn its cost back.

## Where to take this

- **Measure whether the filter helps.** It narrows results, which is not the same as improving
  them. [`retrieval/02`](../retrieval/02_which_parts_helped.ipynb) shows the apparatus, and the
  answer there was that a filter helped a little on average while badly hurting a few queries.
- **Keep the copies honest.** A re-extraction should rewrite the chunks it affects. That is a
  diff driven by the terms document's `extracted_at`, not a full rebuild, and it is the part
  most likely to rot quietly.
- **Extract fields worth filtering on.** Renewal notice deadlines and termination windows are
  what turn this from search into something that tells you what is about to expire.
- **Ask whether the chunk is the right unit.** Clause boundaries would beat fixed 1,200-character
  windows here, and the ground truth to check that with is already in the repo —
  [`flows/01`](../flows/01_rag_that_you_can_trust.ipynb) scores retrieval against
  lawyer-annotated spans.

> **On Couchbase AI Data Plane** — chunking, embedding and keeping the copied-down fields in
> step are three jobs this notebook does by hand and re-does whenever a contract changes. The
> managed service is aimed at exactly that maintenance. What it does not decide for you is
> which fields belong on the chunk, and that is the modelling question this notebook is
> actually about.

In [12]:
# Everything notebooks 01 and 02 created, in one scope. Uncomment to remove it.
# from cbnb.couchbase_io import drop_demo_data
# drop_demo_data(cluster, BUCKET, SCOPE)